In [ ]:
# !pip install duckdb
# !pip install sqlalchemy
# !pip install pyodbc
# !pip install pandas
# !pip install polars


In [3]:
import datetime 
from MODULOS.conexion_sql import get_connection
from MODULOS.conexion_sql import get_sqlalchemy_engine
import duckdb
import pandas as pd

In [4]:
def SQLServer_to_Parquet(ScriptSQL, ruta_parquet):
    
    conn = get_connection()
    if conn is None:
        print("No se pudo conectar a SQL Server")
        return
    try:
        query = f"""
                {ScriptSQL}
        """
        df = pd.read_sql(query, conn)
        print(f"Registros obtenidos: {len(df):,}")
        duckdb.sql(f"""
            COPY (
                SELECT *
                FROM df
            )
            TO '{ruta_parquet}'
            (
                FORMAT PARQUET,
                COMPRESSION ZSTD
            )
        """)
        print("Parquet generado correctamente")
        print(ruta_parquet)
    except Exception as e:
        print(f"Error: {e}")
    finally:
        conn.close()

In [ ]:
# RUTA_PARQUET = "D:/Datos/PAR_ESTADOSENVIO.parquet"
# SCRIPT_SQL = "SELECT * FROM PAR.ESTADOSENVIO WITH(NOLOCK)"
# SQLServer_to_Parquet(
#     SCRIPT_SQL,
#     RUTA_PARQUET
# )

# df_resultado = duckdb.sql(f"""
#     SELECT *
#     FROM '{RUTA_PARQUET}'
#     LIMIT 5
# """).df()
# display(df_resultado)

Conexión establecida con SQL Server


C:\Users\john.cruz\AppData\Local\Temp\ipykernel_20748\917111132.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Registros obtenidos: 8
Parquet generado correctamente
D:/Datos/PAR_ESTADOSENVIO.parquet


,ID_ESTADOENVIO,NOMBRE_ESTADOENVIO,ID_EST_ESTADOENVIO,LOGIN_EMPLEADO,FECHA_CREACION,FECHA_MODIFICACION,msrepl_tran_version,FECHA_EXTRACCION_DW
0,1,RECIBIDO DEL CLIENTE,1,biancd,2002-04-05 08:57:02.000,2018-08-13 15:32:36.693,328D888E-A658-49FC-8FD7-67674CBE35C1,2026-08-28 00:16:00
1,2,EN PROCESAMIENTO,1,biancd,2002-04-05 08:57:30.000,2018-08-13 15:33:43.910,8B91DCCD-DA24-47D6-A2D9-79D2701560C0,2026-08-28 00:16:00
2,3,ENTREGADO,1,biancd,2002-04-05 08:57:53.000,2018-08-13 15:33:23.067,26C342A9-DF39-4FDE-9A09-4F4399BE1B9B,2026-08-28 00:16:00
3,4,ENTREGADO A REMITENTE,1,biancd,2007-11-06 11:53:16.730,2018-08-13 15:34:01.820,08494EAD-715C-48DB-9202-C060A0D36ADC,2026-08-28 00:16:00
4,5,SINIESTRADO,1,biancd,2010-11-16 15:04:11.167,2018-08-13 15:34:22.197,922FF635-756B-46F5-8B59-4B9DDD45D5D5,2026-08-28 00:16:00


In [9]:
ARCHIVO_PARQUET = "LOG_SEGUIMIENTO_ENVIO"

periodos = [
    202605,
    202604,
    202603,
    202602,
    202601
]

for periodo in periodos:
    print(f"Generando parquet para el periodo: {periodo}")
    RUTA_PARQUET = f"D:/Datos/{ARCHIVO_PARQUET}/{ARCHIVO_PARQUET}-{periodo}.parquet"
    
    # Se agregó la 'f' al inicio de la triple comilla
    SCRIPT_SQL = f"""
            DECLARE @FECHAID INT = {periodo};
            -- Se usa CAST(@FECHAID AS VARCHAR(6)) para asegurar el formato correcto antes de concatenar '01'
            DECLARE @FECHA_INI DATETIME = CONVERT(DATETIME, CONCAT(CAST(@FECHAID AS VARCHAR(6)), '01'));
            DECLARE @FECHA_FIN DATETIME = DATEADD(MONTH, 1, @FECHA_INI);
            
            SELECT 
                * 
            FROM LOG.SEGUIMIENTO_ENVIO WITH(NOLOCK) 
            WHERE 
                [ID_PROCESO] IN (5, 6, 9)
                AND FECHA_HORA_MOVIMIENTO >= @FECHA_INI 
                AND FECHA_HORA_MOVIMIENTO < @FECHA_FIN
            OPTION(RECOMPILE);
    """
    
    SQLServer_to_Parquet(
        SCRIPT_SQL,
        RUTA_PARQUET
    )
    print(f"Finaliza periodo: {periodo}")


Generando parquet para el periodo: 202605
Conexión establecida con SQL Server


C:\Users\john.cruz\AppData\Local\Temp\ipykernel_12260\917111132.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Registros obtenidos: 4,185,047
Parquet generado correctamente
D:/Datos/LOG_SEGUIMIENTO_ENVIO/LOG_SEGUIMIENTO_ENVIO-202605.parquet
Finaliza periodo: 202605
Generando parquet para el periodo: 202604
Conexión establecida con SQL Server


C:\Users\john.cruz\AppData\Local\Temp\ipykernel_12260\917111132.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Registros obtenidos: 4,265,426
Parquet generado correctamente
D:/Datos/LOG_SEGUIMIENTO_ENVIO/LOG_SEGUIMIENTO_ENVIO-202604.parquet
Finaliza periodo: 202604
Generando parquet para el periodo: 202603
Conexión establecida con SQL Server


...

Registros obtenidos: 4,771,933
Parquet generado correctamente
D:/Datos/LOG_SEGUIMIENTO_ENVIO/LOG_SEGUIMIENTO_ENVIO-202603.parquet
Finaliza periodo: 202603
Generando parquet para el periodo: 202602
Conexión establecida con SQL Server


C:\Users\john.cruz\AppData\Local\Temp\ipykernel_12260\917111132.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Registros obtenidos: 4,640,043
Parquet generado correctamente
D:/Datos/LOG_SEGUIMIENTO_ENVIO/LOG_SEGUIMIENTO_ENVIO-202602.parquet
Finaliza periodo: 202602
Generando parquet para el periodo: 202601
Conexión establecida con SQL Server


C:\Users\john.cruz\AppData\Local\Temp\ipykernel_12260\917111132.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Registros obtenidos: 4,975,983
Parquet generado correctamente
D:/Datos/LOG_SEGUIMIENTO_ENVIO/LOG_SEGUIMIENTO_ENVIO-202601.parquet
Finaliza periodo: 202601


In [ ]:
df_resultado = duckdb.sql(f"""
    SELECT *
    FROM '{RUTA_PARQUET}'
    LIMIT 5
""").df()
display(df_resultado)

In [10]:
ARCHIVO_PARQUET = "LOG_GUIAS"

periodos = [
    202608,
    202607,
    202606,
    202605,
    202604,
    202603,
    202602,
    202601
]

for periodo in periodos:
    print(f"Generando parquet para el periodo: {periodo}")
    RUTA_PARQUET = f"D:/Datos/{ARCHIVO_PARQUET}/{ARCHIVO_PARQUET}-{periodo}.parquet"
    
    # Se agregó la 'f' al inicio de la triple comilla
    SCRIPT_SQL = f"""
            DECLARE @FECHAID INT = {periodo};
            -- Se usa CAST(@FECHAID AS VARCHAR(6)) para asegurar el formato correcto antes de concatenar '01'
            DECLARE @FECHA_INI DATETIME = CONVERT(DATETIME, CONCAT(CAST(@FECHAID AS VARCHAR(6)), '01'));
            DECLARE @FECHA_FIN DATETIME = DATEADD(MONTH, 1, @FECHA_INI);
            
            SELECT 
                * 
            FROM LOG.GUIAS WITH(NOLOCK) 
            WHERE 
                    FECHA_ENVIO >= @FECHA_INI 
                AND FECHA_ENVIO < @FECHA_FIN
            OPTION(RECOMPILE);
    """
    
    SQLServer_to_Parquet(
        SCRIPT_SQL,
        RUTA_PARQUET
    )
    print(f"Finaliza periodo: {periodo}")


Generando parquet para el periodo: 202608
Conexión establecida con SQL Server


C:\Users\john.cruz\AppData\Local\Temp\ipykernel_12260\917111132.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Registros obtenidos: 1,675,369
Parquet generado correctamente
D:/Datos/LOG_GUIAS/LOG_GUIAS-202608.parquet
Finaliza periodo: 202608
Generando parquet para el periodo: 202607
Conexión establecida con SQL Server


C:\Users\john.cruz\AppData\Local\Temp\ipykernel_12260\917111132.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Registros obtenidos: 2,117,281
Parquet generado correctamente
D:/Datos/LOG_GUIAS/LOG_GUIAS-202607.parquet
Finaliza periodo: 202607
Generando parquet para el periodo: 202606
Conexión establecida con SQL Server


...

Registros obtenidos: 2,204,114
Parquet generado correctamente
D:/Datos/LOG_GUIAS/LOG_GUIAS-202603.parquet
Finaliza periodo: 202603
Generando parquet para el periodo: 202602
Conexión establecida con SQL Server


C:\Users\john.cruz\AppData\Local\Temp\ipykernel_12260\917111132.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Registros obtenidos: 2,338,993
Parquet generado correctamente
D:/Datos/LOG_GUIAS/LOG_GUIAS-202602.parquet
Finaliza periodo: 202602
Generando parquet para el periodo: 202601
Conexión establecida con SQL Server


C:\Users\john.cruz\AppData\Local\Temp\ipykernel_12260\917111132.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Registros obtenidos: 2,429,918
Parquet generado correctamente
D:/Datos/LOG_GUIAS/LOG_GUIAS-202601.parquet
Finaliza periodo: 202601


In [11]:
ARCHIVO_PARQUET = "LOG_DOCUMENTOS"

periodos = [
    202608,
    202607,
    202606,
    202605,
    202604,
    202603,
    202602,
    202601
]

for periodo in periodos:
    print(f"Generando parquet para el periodo: {periodo}")
    RUTA_PARQUET = f"D:/Datos/{ARCHIVO_PARQUET}/{ARCHIVO_PARQUET}-{periodo}.parquet"
    
    # Se agregó la 'f' al inicio de la triple comilla
    SCRIPT_SQL = f"""
            DECLARE @FECHAID INT = {periodo};
            -- Se usa CAST(@FECHAID AS VARCHAR(6)) para asegurar el formato correcto antes de concatenar '01'
            DECLARE @FECHA_INI DATETIME = CONVERT(DATETIME, CONCAT(CAST(@FECHAID AS VARCHAR(6)), '01'));
            DECLARE @FECHA_FIN DATETIME = DATEADD(MONTH, 1, @FECHA_INI);
            
            SELECT 
                * 
            FROM LOG.DOCUMENTOS WITH(NOLOCK) 
            WHERE 
                    FECHA_HORA_MOVIMI >= @FECHA_INI 
                AND FECHA_HORA_MOVIMI < @FECHA_FIN
            OPTION(RECOMPILE);
    """
    
    SQLServer_to_Parquet(
        SCRIPT_SQL,
        RUTA_PARQUET
    )
    print(f"Finaliza periodo: {periodo}")


Generando parquet para el periodo: 202608
Conexión establecida con SQL Server


C:\Users\john.cruz\AppData\Local\Temp\ipykernel_12260\917111132.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Registros obtenidos: 780,346
Parquet generado correctamente
D:/Datos/LOG_DOCUMENTOS/LOG_DOCUMENTOS-202608.parquet
Finaliza periodo: 202608
Generando parquet para el periodo: 202607
Conexión establecida con SQL Server


C:\Users\john.cruz\AppData\Local\Temp\ipykernel_12260\917111132.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Registros obtenidos: 948,257
Parquet generado correctamente
D:/Datos/LOG_DOCUMENTOS/LOG_DOCUMENTOS-202607.parquet
Finaliza periodo: 202607
Generando parquet para el periodo: 202606
Conexión establecida con SQL Server


...

Registros obtenidos: 1,011,325
Parquet generado correctamente
D:/Datos/LOG_DOCUMENTOS/LOG_DOCUMENTOS-202603.parquet
Finaliza periodo: 202603
Generando parquet para el periodo: 202602
Conexión establecida con SQL Server


C:\Users\john.cruz\AppData\Local\Temp\ipykernel_12260\917111132.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Registros obtenidos: 984,023
Parquet generado correctamente
D:/Datos/LOG_DOCUMENTOS/LOG_DOCUMENTOS-202602.parquet
Finaliza periodo: 202602
Generando parquet para el periodo: 202601
Conexión establecida con SQL Server


C:\Users\john.cruz\AppData\Local\Temp\ipykernel_12260\917111132.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Registros obtenidos: 1,001,031
Parquet generado correctamente
D:/Datos/LOG_DOCUMENTOS/LOG_DOCUMENTOS-202601.parquet
Finaliza periodo: 202601


In [ ]:
#  PARAMETRICAS GENERALES BASICAS 
ARCHIVO_PARQUET = "PARAMETRICAS"

TBL_PARAMETRICAS = [
 'PAR.REGIONALES_PAIS'
,'PAR.CIUDADES'
,'PAR.ESTADOSENVIO'
,'PAR.PROCESOS'
,'PAR.ESTADOSGUIA'
,'PAR.ZONASURBANAS'
,'DIM.VW_TIEMPO_CALENDARIO_Y_FESTIVOS'
]

for tabla in TBL_PARAMETRICAS:
    print(f"Generando Tabla: {tabla}")
    RUTA_PARQUET = f"D:/Datos/{ARCHIVO_PARQUET}/{tabla}.parquet"
    SCRIPT_SQL = f"""
            SELECT 
                * 
            FROM {tabla} WITH(NOLOCK) 
            OPTION(RECOMPILE);
    """
    
    SQLServer_to_Parquet(
        SCRIPT_SQL,
        RUTA_PARQUET
    )
    print(f"Finaliza carga de {tabla}: {RUTA_PARQUET}")


Generando Tabla: PAR.REGIONALES_PAIS
Conexión establecida con SQL Server
Registros obtenidos: 11
Parquet generado correctamente
D:/Datos/PARAMETRICAS/PAR.REGIONALES_PAIS.parquet
Finaliza carga de PAR.REGIONALES_PAIS: D:/Datos/PARAMETRICAS/PAR.REGIONALES_PAIS.parquet
Generando Tabla: PAR.CIUDADES
Conexión establecida con SQL Server


C:\Users\john.cruz\AppData\Local\Temp\ipykernel_29036\917111132.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)
C:\Users\john.cruz\AppData\Local\Temp\ipykernel_29036\917111132.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Registros obtenidos: 9,625
Parquet generado correctamente
D:/Datos/PARAMETRICAS/PAR.CIUDADES.parquet
Finaliza carga de PAR.CIUDADES: D:/Datos/PARAMETRICAS/PAR.CIUDADES.parquet
Generando Tabla: PAR.ESTADOSENVIO
Conexión establecida con SQL Server
Registros obtenidos: 8
Parquet generado correctamente
D:/Datos/PARAMETRICAS/PAR.ESTADOSENVIO.parquet
Finaliza carga de PAR.ESTADOSENVIO: D:/Datos/PARAMETRICAS/PAR.ESTADOSENVIO.parquet
Generando Tabla: PAR.PROCESOS
Conexión establecida con SQL Server
Registros obtenidos: 52
Parquet generado correctamente
D:/Datos/PARAMETRICAS/PAR.PROCESOS.parquet
Finaliza carga de PAR.PROCESOS: D:/Datos/PARAMETRICAS/PAR.PROCESOS.parquet
Generando Tabla: PAR.ESTADOSGUIA
Conexión establecida con SQL Server
Registros obtenidos: 12
Parquet generado correctamente
D:/Datos/PARAMETRICAS/PAR.ESTADOSGUIA.parquet
Finaliza carga de PAR.ESTADOSGUIA: D:/Datos/PARAMETRICAS/PAR.ESTADOSGUIA.parquet
Generando Tabla: PAR.ZONASURBANAS
Conexión establecida con SQL Server


C:\Users\john.cruz\AppData\Local\Temp\ipykernel_29036\917111132.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)
C:\Users\john.cruz\AppData\Local\Temp\ipykernel_29036\917111132.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)
C:\Users\john.cruz\AppData\Local\Temp\ipykernel_29036\917111132.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)
C:\Users\john.cruz\AppData\Local\Temp\ipykernel_29036\917111132.py:11: UserWarning: pand

Registros obtenidos: 25,525
Parquet generado correctamente
D:/Datos/PARAMETRICAS/PAR.ZONASURBANAS.parquet
Finaliza carga de PAR.ZONASURBANAS: D:/Datos/PARAMETRICAS/PAR.ZONASURBANAS.parquet
Generando Tabla: DIM.VW_TIEMPO_CALENDARIO_Y_FESTIVOS
Conexión establecida con SQL Server


C:\Users\john.cruz\AppData\Local\Temp\ipykernel_29036\917111132.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Registros obtenidos: 17,726
Parquet generado correctamente
D:/Datos/PARAMETRICAS/DIM.VW_TIEMPO_CALENDARIO_Y_FESTIVOS.parquet
Finaliza carga de DIM.VW_TIEMPO_CALENDARIO_Y_FESTIVOS: D:/Datos/PARAMETRICAS/DIM.VW_TIEMPO_CALENDARIO_Y_FESTIVOS.parquet


In [5]:
#  PARAMETRICAS RED OPERATIVA
ARCHIVO_PARQUET = "PARAMETRICAS"
NOMBRE_TABLA = "PAR.RED_OPERATIVA_OFERTA"
SQLServer_Script = f"""
SELECT * FROM [TMP].[MPCCO_VJ_PARAMETROS_RED_OPERATIVA] WITH(NOLOCK)
"""

print(f"Generando Tabla: {NOMBRE_TABLA}")
RUTA_PARQUET = f"D:/Datos/{ARCHIVO_PARQUET}/{NOMBRE_TABLA}.parquet"
SCRIPT_SQL = SQLServer_Script

SQLServer_to_Parquet(
    SCRIPT_SQL,
    RUTA_PARQUET
)
print(f"Finaliza carga de {NOMBRE_TABLA}: {RUTA_PARQUET}")

Generando Tabla: PAR.RED_OPERATIVA_OFERTA
Conexión establecida con SQL Server


C:\Users\john.cruz\AppData\Local\Temp\ipykernel_4552\917111132.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Registros obtenidos: 5,606,406
Parquet generado correctamente
D:/Datos/PARAMETRICAS/PAR.RED_OPERATIVA_OFERTA.parquet
Finaliza carga de PAR.RED_OPERATIVA_OFERTA: D:/Datos/PARAMETRICAS/PAR.RED_OPERATIVA_OFERTA.parquet
